# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YassinKhodir/yassin-flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the held-out-client score into a small, human-reviewed playbook. The model orders review candidates; it does not edit, publish, remove, merge, or refresh content.

## 1. Ranked actions and reason codes

The score uses the same five early-March features and grouped split as ML-09. I rank only the held-out rows so the exported queue stays tied to an out-of-sample result.

| Archetype / reason code | Suggested action | Why it appears |
|---|---|---|
| `visible_low_ctr` | Review title, snippet, and intent match | The page has visibility but captures few clicks |
| `high_volume_position_21_50` | Review content depth and internal links | The page has demand and room to move |
| `position_unstable` | Monitor before editing | The average position may hide a noisy pattern |
| `general_decline_risk` | Manual diagnosis | The model score is high but no single rule explains it |

Actions are suggestions for an editor, never automatic instructions.

In [1]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

required = {
    "duckdb": "duckdb", "pandas": "pandas", "numpy": "numpy",
    "sklearn": "scikit-learn", "matplotlib": "matplotlib",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import duckdb
import numpy as np
import pandas as pd
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data_root = os.environ.get("FLYRANK_DATA_ROOT")
if data_root:
    REL = data_root.rstrip("/")
else:
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            hf_token = None
    if not hf_token:
        raise RuntimeError("Add a Hugging Face READ token as a Colab Secret named HF_TOKEN; never paste it into this public notebook.")
    REL = "hf://datasets/FlyRank/internship-warehouse"

con = duckdb.connect()
if not data_root:
    escaped_token = hf_token.replace("'", "''")
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{escaped_token}')")
    del hf_token, escaped_token

MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
feature_query = f"""
WITH early AS (
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS early_impressions,
        SUM(gsc_clicks) AS early_clicks,
        COUNT(*) AS early_observed_days,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS early_active_days,
        SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position IS NOT NULL
                 THEN gsc_avg_position * gsc_impressions ELSE 0 END)
          / NULLIF(SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position IS NOT NULL
                            THEN gsc_impressions ELSE 0 END), 0) AS early_avg_position,
        STDDEV_SAMP(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position END) AS early_position_volatility
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
),
late AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS late_impressions,
           COUNT(*) AS late_observed_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    GROUP BY 1, 2
)
SELECT
    e.client_hash_id, e.content_hash_id,
    LN(1 + e.early_impressions) AS log_early_impressions,
    1.0 * e.early_clicks / NULLIF(e.early_impressions, 0) AS early_ctr,
    e.early_active_days, e.early_avg_position, e.early_position_volatility,
    e.early_impressions,
    CAST((1.0 * l.late_impressions / NULLIF(l.late_observed_days, 0))
         < 0.8 * (1.0 * e.early_impressions / NULLIF(e.early_observed_days, 0)) AS INTEGER)
      AS is_future_decline
FROM early e
JOIN late l USING (client_hash_id, content_hash_id)
ORDER BY e.client_hash_id, e.content_hash_id
"""

frame = con.sql(feature_query).df()
features = [
    "log_early_impressions", "early_ctr", "early_active_days",
    "early_avg_position", "early_position_volatility",
]
print(f"Rows: {len(frame):,}; clients: {frame['client_hash_id'].nunique()}; sklearn: {sklearn.__version__}")
import matplotlib.pyplot as plt

Rows: 77,400; clients: 38; sklearn: 1.8.0


In [2]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(frame, frame["is_future_decline"], groups=frame["client_hash_id"]))
train = frame.iloc[train_idx].copy()
queue = frame.iloc[test_idx].copy()
assert set(train["client_hash_id"]).isdisjoint(set(queue["client_hash_id"]))

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)
model.fit(train[features], train["is_future_decline"])
queue["review_score"] = model.predict_proba(queue[features])[:, 1]

high_cut = queue["review_score"].quantile(0.95)
medium_cut = queue["review_score"].quantile(0.80)
queue["priority"] = np.select(
    [queue["review_score"] >= high_cut, queue["review_score"] >= medium_cut],
    ["high", "medium"], default="monitor",
)
queue["reason_code"] = np.select(
    [
        (queue["early_impressions"] >= 1000) & (queue["early_ctr"] < 0.01),
        (queue["early_impressions"] >= 1000) & queue["early_avg_position"].between(20, 50, inclusive="right"),
        queue["early_position_volatility"].fillna(0) >= 10,
    ],
    ["visible_low_ctr", "high_volume_position_21_50", "position_unstable"],
    default="general_decline_risk",
)
queue["suggested_action"] = np.select(
    [
        queue["priority"] == "monitor",
        queue["reason_code"] == "visible_low_ctr",
        queue["reason_code"] == "high_volume_position_21_50",
        queue["reason_code"] == "position_unstable",
    ],
    ["monitor", "review title, snippet, and intent", "review depth and internal links", "wait for a steadier window"],
    default="manual decline diagnosis",
)
queue = queue.sort_values(
    ["review_score", "early_impressions", "content_hash_id"], ascending=[False, False, True]
).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

print(f"Held-out queue: {len(queue):,} rows; high cutoff: {high_cut:.3f}; medium cutoff: {medium_cut:.3f}")
print(queue.groupby(["priority", "reason_code"]).size().rename("n").reset_index().to_string(index=False))

Held-out queue: 15,776 rows; high cutoff: 0.456; medium cutoff: 0.420
priority                reason_code    n
    high       general_decline_risk  354
    high          position_unstable  160
    high            visible_low_ctr  275
  medium       general_decline_risk 1268
  medium          position_unstable  577
  medium            visible_low_ctr  522
 monitor       general_decline_risk 8523
 monitor high_volume_position_21_50    7
 monitor          position_unstable  813
 monitor            visible_low_ctr 3277


## 2. Intended use, decay insight, value, and limits

**Intended user:** an SEO or content editor planning a review batch. Start with high-priority rows, inspect the evidence, estimate business value and editing cost, then choose refresh, rewrite, merge, protect, monitor, or no action.

**Observed decay insight:** pages in positions 21–50 had a 40.7% later-decline rate in the March signal audit, compared with the overall 34.5% rate. The grouped model was weak overall (ROC-AUC 0.510), so this is a directional review cue, not proof that pages decay because of position or that refresh will recover them.

**Cost/value rule:** high score plus meaningful search demand can justify a short review; expensive rewrites need stronger evidence such as strategic value, stable demand, correct intent, and a clear content gap.

**Limits:** one mid-panel month, 38 represented clients in the full frame, 10 held-out clients in validation, an outcome-defined eligible population, no query text or business context, and no controlled refresh experiment.

In [3]:
playbook_summary = pd.DataFrame([
    ["high", "review first", "editor checks demand, intent, quality, cannibalization, and cost"],
    ["medium", "review if capacity remains", "editor confirms the pattern is stable"],
    ["monitor", "no immediate edit", "wait for another complete measurement window"],
], columns=["priority", "default_use", "human_gate"])
playbook_summary

  priority  ...                                         human_gate
0     high  ...  editor checks demand, intent, quality, canniba...
1   medium  ...              editor confirms the pattern is stable
2  monitor  ...       wait for another complete measurement window

[3 rows x 3 columns]

## 3. Human review and the no-go list

Before acting, a person must check:

- whether demand is stable or seasonal;
- whether the page matches the intended query and business goal;
- whether another page already owns the topic;
- whether a technical/indexing problem explains the decline;
- whether the content is accurate, current, and worth the editing cost;
- whether the page is already scheduled for migration, merge, or removal.

**Do not automate:** publishing edits, deleting or pruning pages, merging URLs, redirects, metadata changes, outreach, recrawl requests, or claims that a refresh will improve rank. The queue may order attention; humans own every external action.

In [4]:
review_fields = [
    "stable_demand", "intent_match", "business_value", "content_gap",
    "cannibalization_check", "technical_check", "estimated_edit_cost", "editor_decision",
]
no_go_actions = [
    "publish", "delete", "merge", "redirect", "change_metadata", "request_recrawl",
]
print("Required review fields:", len(review_fields))
print("Actions that remain manual:", ", ".join(no_go_actions))
print("Model-triggered external actions: 0")

Required review fields: 8
Actions that remain manual: publish, delete, merge, redirect, change_metadata, request_recrawl
Model-triggered external actions: 0


## 4. Monitoring and retrain triggers

Review the queue monthly, but retrain only when enough new labeled data exists. Investigate or pause the score if any of these occur:

- the eligible-page count changes by more than 20%;
- held-out base rate shifts by more than 10 percentage points;
- precision@100 falls below the current held-out base rate for two runs;
- more than 10% of required feature values are missing;
- a source definition, tracking flag, or measurement window changes;
- editors reject more than half of the high-priority rows in two review batches.

The response to a trigger is diagnosis first, not automatic retraining.

In [5]:
monitoring_rules = pd.DataFrame([
    ["eligible row count", "change > 20%", "check filters and source coverage"],
    ["base rate", "shift > 10 percentage points", "review label and seasonality"],
    ["precision@100", "below base rate twice", "pause model ranking"],
    ["feature missingness", "> 10%", "inspect tracking availability"],
    ["editor rejection", "> 50% twice", "review reasons and thresholds"],
], columns=["signal", "trigger", "response"])
monitoring_rules

                signal  ...                           response
0   eligible row count  ...  check filters and source coverage
1            base rate  ...       review label and seasonality
2        precision@100  ...                pause model ranking
3  feature missingness  ...      inspect tracking availability
4     editor rejection  ...      review reasons and thresholds

[5 rows x 3 columns]

## 5. Exports for the paper

The CSV is regenerated and remains outside git. The aggregate JSON and comparison figure are safe to commit and let the paper trace its headline numbers.

In [6]:
output_dir = Path("../outputs") if Path.cwd().name == "notebooks" else Path("work/outputs")
figure_dir = Path("../figures") if Path.cwd().name == "notebooks" else Path("work/figures")
output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

export_columns = [
    "rank", "client_hash_id", "content_hash_id", "review_score", "priority",
    "reason_code", "suggested_action", "early_impressions", "early_ctr",
    "early_active_days", "early_avg_position", "early_position_volatility",
]
queue[export_columns].to_csv(output_dir / "validated_action_queue.csv", index=False)

metrics = {
    "queue_rows": int(len(queue)),
    "held_out_clients": int(queue["client_hash_id"].nunique()),
    "high_priority_rows": int((queue["priority"] == "high").sum()),
    "medium_priority_rows": int((queue["priority"] == "medium").sum()),
    "monitor_rows": int((queue["priority"] == "monitor").sum()),
    "high_priority_cutoff": round(float(high_cut), 6),
    "medium_priority_cutoff": round(float(medium_cut), 6),
    "grouped_model_roc_auc": round(float(roc_auc_score(queue["is_future_decline"], queue["review_score"])), 6),
}
with open(output_dir / "ml10_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

chart_values = [0.233, 0.150, 0.310]
fig, ax = plt.subplots(figsize=(6.5, 3.8))
bars = ax.bar(["Held-out base rate", "Frozen rule", "Logistic model"], chart_values,
              color=["#9ca3af", "#7c3aed", "#16a34a"])
ax.set_ylabel("Precision@100")
ax.set_ylim(0, 0.36)
ax.set_title("Top-100 decline capture on held-out clients")
ax.bar_label(bars, labels=[f"{v:.3f}" for v in chart_values], padding=3)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(figure_dir / "heldout_precision_at_100.png", dpi=180)
plt.show()

print(f"Wrote {len(queue):,} queue rows and aggregate metrics.")
print(pd.Series(metrics).to_string())

Wrote 15,776 queue rows and aggregate metrics.
queue_rows                15776.000000
held_out_clients             10.000000
high_priority_rows          789.000000
medium_priority_rows       2367.000000
monitor_rows              12620.000000
high_priority_cutoff          0.455889
medium_priority_cutoff        0.420422
grouped_model_roc_auc         0.509776


## Self-check

- [x] Ranked actions have reason codes and clear human owners
- [x] Archetypes map to practical actions
- [x] Intended use, cost/value, decay insight, and limits are stated
- [x] Human-review gates and no-go actions are explicit
- [x] Monitoring and retrain triggers are measurable
- [x] The queue, aggregate receipt, and paper figure are exported
- [x] No automated content change is recommended
- [x] Claims stay observed, directional, and decision-support only
- [x] The notebook runs top to bottom without errors